In [2]:
import os
import pymysql
import pandas as pd

In [3]:
conn = pymysql.connect(
    host=os.getenv('MYSQL_HOST'),
    user=os.getenv('MYSQL_USER'),
    password=os.getenv('MYSQL_PASSWORD'),
    database=os.getenv('MYSQL_DATABASE')
)

# UC-1

In [4]:
# Q1
full_join_query = """SELECT count(*)
FROM beers
WHERE TRUE
;"""
df = pd.read_sql_query(full_join_query, con=conn)
df

/tmp/ipykernel_2195/2686738455.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(full_join_query, con=conn)


,count(*)
0,5901


In [5]:
# Q2
full_join_query = """SELECT brew.name as brewery, count(*)
FROM beers
JOIN breweries as brew on brew.id = beers.brewery_id
WHERE TRUE
GROUP BY brew.name
ORDER BY 2 DESC
LIMIT 10
;"""
df = pd.read_sql_query(full_join_query, con=conn)
df

/tmp/ipykernel_2195/2009646827.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(full_join_query, con=conn)


,brewery,count(*)
0,Midnight Sun Brewing Co.,57
1,Rogue Ales,49
2,Anheuser-Busch,38
3,Troegs Brewing,37
4,Egan Brewing,37
5,Boston Beer Company,36
6,Titletown Brewing,34
7,F.X. Matt Brewing,34
8,Sierra Nevada Brewing Co.,33
9,Stone Brewing Co.,32


In [6]:
# Q3
full_join_query = """SELECT beers.name, brew.name, abv
FROM beers
JOIN breweries as brew on brew.id = beers.brewery_id
WHERE TRUE
AND brew.country = 'France'
ORDER BY 3 DESC
LIMIT 10
;"""
df = pd.read_sql_query(full_join_query, con=conn)
df

/tmp/ipykernel_2195/265909448.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(full_join_query, con=conn)


,name,name,abv
0,Belzebuth,Brasserie Grain D'Orge,13.0
1,3 Monts,Brasserie De Saint Sylvestre,8.5
2,Gavroche French Red Ale,Brasserie De Saint Sylvestre,8.5
3,Yeti,Brasserie des Cimes,8.0
4,Jenlain Blonde,Brasserie Duyck,7.5
5,Blonde,Brasserie La Choulette,7.5
6,Les Sans Culottes,Brasserie La Choulette,7.0
7,Framboise,Brasserie La Choulette,6.0
8,Jenlain St Druon de Sebourg,Brasserie Duyck,6.0
9,Castelain St.Amand French Country Ale,Brasserie Bnifontaine,5.9


In [7]:
# Q4
full_join_query = """SELECT brew.country, count(*) as nb_porter, AVG(ABV) as abv_mean
FROM beers
JOIN breweries as brew on brew.id = beers.brewery_id
JOIN styles on styles.id = beers.style_id
WHERE TRUE
AND styles.style_name = 'Porter'
GROUP BY brew.country
ORDER BY 2 DESC
;"""
df = pd.read_sql_query(full_join_query, con=conn)
df

/tmp/ipykernel_2195/2558923092.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(full_join_query, con=conn)


,country,nb_porter,abv_mean
0,United States,234,2.330556
1,United Kingdom,14,3.950000
2,Canada,5,0.000000
3,Sweden,1,5.500000
4,Poland,1,8.300000
5,Denmark,1,8.000000
6,Norway,1,7.000000
7,Germany,1,7.100000
8,Switzerland,1,4.500000
9,Czech Republic,1,8.000000


In [8]:
# Q5 - observation : certaines opérations "simples" sont un peu compliquées à réaliser en SQL
q = """
WITH country_cnt AS (
    SELECT 
        brew.country AS country,
        COUNT(*) AS cnt
    FROM beers
    JOIN breweries AS brew ON brew.id = beers.brewery_id
    GROUP BY brew.country
    ORDER BY cnt DESC
), ranked_countries AS (
    SELECT
        country, cnt, ROW_NUMBER() OVER (ORDER BY cnt) as rnk
    FROM country_cnt
), nlines AS (
    SELECT count(*) as nn
    FROM country_cnt
), proxy_median AS (
    SELECT 
        country, cnt, POWER((rnk / nn) - 1/2, 2) as proxmed
    FROM ranked_countries
    LEFT JOIN nlines ON TRUE
)
SELECT * 
FROM proxy_median
ORDER BY proxmed ASC
LIMIT 1
;"""
df = pd.read_sql_query(q, con=conn)
df

/tmp/ipykernel_2195/3545058473.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(q, con=conn)


,country,cnt,proxmed
0,Jamaica,3,0.000067


# UC-2
Exo difficile

In [9]:
q = f"""
WITH clicked_pos AS (
    SELECT query, user_id, pos_in_serp as clicked_pos_in_serp
    FROM `beers_feedback` 
    WHERE TRUE
    AND clicked_id = id_in_serp
), 
db_and_clicked_and_seen AS (
    SELECT 
        beers_feedback.*, 
        CASE WHEN pos_in_serp <= clicked_pos_in_serp THEN 1 ELSE 0 END as seen,
        CASE WHEN pos_in_serp = clicked_pos_in_serp THEN 1 ELSE 0 END as clicked
    FROM beers_feedback
    LEFT JOIN clicked_pos on clicked_pos.user_id = beers_feedback.user_id
),
cascade_probas AS (
    SELECT
        query, id_in_serp, SUM(seen) as n_seen, SUM(clicked) as n_clicked, SUM(clicked)/SUM(seen) as click_proba_cascade
        FROM db_and_clicked_and_seen
        WHERE TRUE
        AND 
            seen = 1
        GROUP BY query, id_in_serp
        ORDER BY query, click_proba_cascade
)
SELECT 
    query, descript, click_proba_cascade
    FROM cascade_probas
    JOIN beers on beers.id = cascade_probas.id_in_serp
    WHERE TRUE
    AND length(descript) > 1
;"""
df = pd.read_sql_query(q, con=conn)
df

/tmp/ipykernel_2195/4267506534.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(q, con=conn)


,query,descript,click_proba_cascade
0,amber ale with caramel flavor,"A refreshing, crisp and clean amber colored la...",0.0000
1,amber ale with caramel flavor,"Eel River Brewing Company, brewers of Californ...",0.0000
2,amber ale with caramel flavor,"A rich, malty beer made with classic American ...",0.1667
3,amber ale with caramel flavor,"Boulevard Pale Ale is a smooth, fruity, well-b...",0.2500
4,amber ale with caramel flavor,A clean crisp ale with a beautiful reddish car...,0.3333
...,...,...,...
326,winter ale with malty flavor,At Weyerbacher we've created a Winter Ale that...,0.0000
327,winter ale with malty flavor,Our ever-changing spiced winter seasonal. A ta...,0.0000
328,winter ale with malty flavor,"""This copper colored ale is smooth, malty, and...",0.2500
329,winter ale with malty flavor,The psycho in the pack... K-9 Cruiser is a dar...,0.3333


# UC-3 search from a query

**Observation :** On ne pourra pas aller bien loin en terme de souplesse dans la requête

In [10]:
QUERY = "stout"

q = f"""
WITH descriptions AS (
    SELECT 
        brew.name as brewery, beers.name as name, CONCAT(beers.descript, brew.descript) as descr
    FROM beers
    JOIN breweries as brew on brew.id = beers.brewery_id
    WHERE TRUE
    AND LENGTH(beers.descript) + LENGTH(beers.descript) > 2
)
SELECT *
FROM descriptions
WHERE True
AND descriptions.descr LIKE '%{QUERY}%'
;"""
pd.read_sql_query(q, con=conn)

/tmp/ipykernel_2195/2478649995.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query(q, con=conn)


,brewery,name,descr
0,Magic Hat,Hocus Pocus,Our take on a classic summer ale. A toast to ...
1,Odell Brewing,Cutthroat Porter,Not quite a stout but definitely no lightweigh...
2,Deschutes Brewery,Buzzsaw Brown,"By 1915, Bend, Oregon, was alive with the soun..."
3,Coopers Brewery,Dark Ale,A dark brew full of promise. Coopers Dark Ale ...
4,Deschutes Brewery,Cinder Cone Red,Cinder Cone Red's diverse selection of hops an...
...,...,...,...
219,Dark Horse Brewing Co.,One Oatmeal Stout,Number one in a series of five stouts produced...
220,Goose Island Beer Company - Clybourn,Bourbon County Brand Coffee Stout,Everyday Goose Island smells the wonderful cof...
221,Central Waters Brewing Company,Brewhouse Coffee Stout,A coffee lover's delight! A wonderful stout in...
222,Nikenjayamochi Kadoya Honten Co.,Ise Kadoya Stout,A very yeasty beer with an earthy aroma. Rich ...


# UC-4 vectorize items

Obligé de sortir de SQL pour faire 

In [ ]:
q = """
WITH data AS (
    SELECT 
        beers.id, beers.name, beers.abv, beers.ibu, beers.srm, beers.descript as beer_descr,
        brew.descript as brewer_descript, brew.name as brewery,
        styles.style_name
    FROM beers
    LEFT JOIN breweries as brew on brew.id = beers.brewery_id
    LEFT JOIN styles on styles.id = beers.style_id
), descriptions AS (
    SELECT 
        id,
        CONCAT('the beer ', name, ' from brewery ', brewery, ' (', brewer_descript, ') crafts the beer ', name, ' defined as ', beer_descr, '. Spec of the beer are: ABV=', abv, ', IBU=', ibu, ', SRM=', srm) as to_vectorize
    FROM data
)
SELECT 
    id, to_vectorize
FROM descriptions
WHERE True
    AND id % 12 = 3
;"""
df = pd.read_sql_query(q, con=conn)

In [ ]:
from typing import List
import requests

def batched(iterable, batch_size=16):
    l = len(iterable)
    for ndx in range(0, l, batch_size):
        yield iterable[ndx:min(ndx + batch_size, l)]

class Vectorizer:
    url = "http://vectorizer:8000/embed"
    
    @staticmethod
    def embed(texts: List[str]):
        return [Vectorizer._embed_one(tt) for tt in texts]
        
    @staticmethod
    def _embed_one(text: str):
        payload = {"text": text}
        
        response = requests.post(Vectorizer.url, json=payload)
        try:
            response.raise_for_status()  # Raise an exception for HTTP errors
            return response.json()["vector"]
        except:
            return None


# UC-5 : answer question in corpa
Pas vraiment de possibilité native en SQL